# Lab | Tools prompting


## Intro

In this lab you'll teach a chat model that has **no native tool-calling support** how to call tools anyway — just by prompting it correctly.

Here's what to expect:

1. **Follow a full worked demo** — We'll walk through every step together: defining `add` and `multiply` tools, writing a prompt that describes them to the model, parsing the model's JSON tool-call output, and invoking the right tool with the right arguments.
2. **Replicate it yourself with new tools** — Then, you'll replace the two demo tools with **3 new tools** of your own choosing and adjust the prompts accordingly, rebuilding the same pipeline so the model can correctly select and invoke your new tools.

By the end of this lab, you'll understand how ad-hoc tool calling works under the hood and be able to apply the pattern to any tools you design.

<br>

## How to add ad-hoc tool calling capability to LLMs and Chat Models

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

<br>

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

<br>

## Install dependencies

Uncomment and run the cells below to install the dependencies required for this notebook.

Tip: Use a virtual environment to keep this project's dependencies isolated from your system Python and other projects.

In [1]:
!pip install "langchain<0.3" "langchain-core<0.3" "langchain-community<0.3" "langchain-openai<0.2"

  Using cached langchain-0.2.17-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl.metadata (6.2 kB)
  Using cached langchain_community-0.2.19-py3-none-any.whl.metadata (2.7 kB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl.metadata (2.6 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl.metadata (2.3 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4-cp313-cp313-macosx_26_0_arm64.whl
  Using cached tenacity-8.5.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached openai-1.109.1-py3-none-any.whl.metadata (29 kB)
  Using cached tiktoken-0.14.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
Using cache

<br>

## Initial Setup



You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [2]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")

<br>

##  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

### Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download)  
   Download and run the installer.

2. **Verify Installation**  
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**  
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**  
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

###  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


<br>

## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [3]:
from langchain_core.tools import tool


@tool
def multiply(x: float, y: float) -> float:
    """Multiply two numbers together."""
    return x * y


@tool
def add(x: int, y: int) -> int:
    "Add two numbers."
    return x + y


tools = [multiply, add]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)

--
multiply
Multiply two numbers together.
{'x': {'title': 'X', 'type': 'number'}, 'y': {'title': 'Y', 'type': 'number'}}
--
add
Add two numbers.
{'x': {'title': 'X', 'type': 'integer'}, 'y': {'title': 'Y', 'type': 'integer'}}


In [4]:
multiply.invoke({"x": 4, "y": 5})

20.0

<br>

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [5]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)

multiply(x: float, y: float) -> float - Multiply two numbers together.
add(x: int, y: int) -> int - Add two numbers.


In [6]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools. 
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use. 
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding 
to the argument names and the values corresponding to the requested values.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

In [7]:
chain = prompt | model
message = chain.invoke({"input": "what's 3 plus 1132"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)

{
  "name": "add",
  "arguments": {
    "x": 3,
    "y": 1132
  }
}


Here' start the solution for the much more difficult instruction:


<br>

## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [8]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "what's thirteen times 4"})

{'name': 'multiply', 'arguments': {'x': 13, 'y': 4}}

<br>

Amazing! 🎉 

We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!


<br>

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke 
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [8]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

<br>

Let's test this out 🧪!

In [9]:
invoke_tool({"name": "multiply", "arguments": {"x": 3, "y": 5}})

15.0

<br>

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [10]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "what's thirteen times 4.14137281"})

53.83784653

<br>

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [11]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "what's thirteen times 4.14137281"})

{'name': 'multiply',
 'arguments': {'x': 13, 'y': 4.14137281},
 'output': 53.83784653}

<br>
<hr>
<br>
<br>

## 🚀 Your turn

Time to make this lab your own!

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information. In reality, if you're using more complex tools, you may start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

Replace the `multiply` and `add` tools with **3 new tools** of your choosing. They don't have to be math-related — a temperature converter, a string reverser, a word counter, a currency converter... pick whatever sounds fun to build.

<br>


Here's what to do:

1. **Design your 3 tools** — Write 3 new functions decorated with `@tool`, each with a clear docstring (this becomes the tool's description shown to the model) and typed arguments.
2. **Update the tools list** — Replace `tools = [multiply, add]` with your 3 new tools.
3. **Regenerate the prompt** — Re-run `render_text_description(tools)` and rebuild `system_prompt`/`prompt` so the model sees the correct tool names, descriptions, and arguments.
4. **Rebuild the chain** — Recreate `chain = prompt | model | JsonOutputParser() | invoke_tool` with your new tools (the `invoke_tool` function already looks tools up dynamically, so it should work as-is).
5. **Put it to the test** — Ask your chain several natural-language questions, one for each tool, and confirm it picks the right tool with the right arguments every time.
6. **Reflect** — In a markdown cell, briefly note any cases where the model picked the wrong tool or malformed the arguments, and why you think that happened.

<br>


💡 **Tip:**

- Print `rendered_tools` after updating your tools list to double check the model is actually seeing the descriptions you expect.

<br>


⭐️ **Bonus ideas:**

- Add a 4th tool that takes more than 2 arguments, or a mix of types (e.g. `str` and `int`), and see if the model still handles it correctly.
- Add a few-shot example to your `system_prompt`: include one sample question together with the exact JSON output you'd want the model to produce for it. This gives the model a concrete pattern to copy, which often makes its answers more consistent.
- Add error handling around `invoke_tool`: if the tool name doesn't exist or the arguments are wrong, catch that error instead of crashing, then send the error message back to the model and ask it to try again with a corrected JSON blob.
- Ask a question that doesn't match any tool at all — how does the model respond, and how could you handle that gracefully in `invoke_tool`?


## ✅ My solution

### Step 1 — Design 3 new tools
- `celsius_to_fahrenheit`: converts a temperature
- `reverse_string`: reverses a piece of text
- `count_words`: counts the words in a text

The **docstring** becomes the description the model reads, so each one says clearly what the tool does and when to use it.

In [12]:
from langchain_core.tools import tool


@tool
def celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature from degrees Celsius to degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


@tool
def reverse_string(text: str) -> str:
    """Reverse the characters of a piece of text, e.g. 'hello' becomes 'olleh'."""
    return text[::-1]


@tool
def count_words(text: str) -> int:
    """Count how many words are in a piece of text."""
    return len(text.split())

### Step 2 — Update the tools list

In [13]:
tools = [celsius_to_fahrenheit, reverse_string, count_words]

# quick sanity check that each tool works on its own
print(celsius_to_fahrenheit.invoke({"celsius": 100}))
print(reverse_string.invoke({"text": "LangChain"}))
print(count_words.invoke({"text": "I am learning tool calling today"}))

212.0
niahCgnaL
6


### Step 3 — Regenerate the prompt
Print `rendered_tools` to check the model sees the new names, descriptions, and arguments.

In [14]:
rendered_tools = render_text_description(tools)
print(rendered_tools)

celsius_to_fahrenheit(celsius: float) -> float - Convert a temperature from degrees Celsius to degrees Fahrenheit.
reverse_string(text: str) -> str - Reverse the characters of a piece of text, e.g. 'hello' becomes 'olleh'.
count_words(text: str) -> int - Count how many words are in a piece of text.


In [15]:
system_prompt = f"""\
You are an assistant that has access to the following set of tools.
Here are the names and descriptions for each tool:

{rendered_tools}

Given the user input, return the name and input of the tool to use.
Return your response as a JSON blob with 'name' and 'arguments' keys.

The `arguments` should be a dictionary, with keys corresponding
to the argument names and the values corresponding to the requested values.

Return ONLY the JSON blob, with no explanation before or after it.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)

### Step 4 — Rebuild the chain
`invoke_tool` looks tools up from the global `tools` list each time it runs, so it picks up the new tools without changes. We use `RunnablePassthrough.assign` so we can see **which tool and arguments** the model chose, plus the output.

In [16]:
chain = prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)

### Step 5 — Put it to the test
One question per tool, plus a few rephrased versions to check consistency.

In [17]:
# Tool 1: temperature
chain.invoke({"input": "What is 25 degrees Celsius in Fahrenheit?"})

{'name': 'celsius_to_fahrenheit', 'arguments': {'celsius': 25}, 'output': 77.0}

In [18]:
# Tool 2: reverse a string
chain.invoke({"input": "Can you reverse the word 'Ironhack'?"})

{'name': 'reverse_string',
 'arguments': {'text': 'Ironhack'},
 'output': 'kcahnorI'}

In [ ]:
# Tool 3: count words
chain.invoke({"input": "How many words are in this sentence: 'The quick brown fox jumps over the lazy dog'?"})

In [ ]:
# Extra tests: rephrased questions to check the model still picks the right tool
test_questions = [
    "It's -10 C outside, how much is that in F?",
    "Write 'hello world' backwards",
    "Count the words in: AI engineers build useful tools",
]

for q in test_questions:
    try:
        result = chain.invoke({"input": q})
        print(f"Q: {q}\n   tool: {result['name']} | args: {result['arguments']} | output: {result['output']}\n")
    except Exception as e:
        print(f"Q: {q}\n   ERROR: {type(e).__name__}: {e}\n")

### Step 6 — Reflection

*(Update these notes after checking your own outputs — phi3 can behave differently on each run.)*

- **Correct choices:** For clear questions ("25 degrees Celsius in Fahrenheit", "reverse the word 'Ironhack'"), the model picked the right tool and the right argument names.
- **Possible problems to look for:**
  - **Extra text around the JSON** (e.g. "Sure! Here is the JSON: ..."). This breaks `JsonOutputParser`. Adding *"Return ONLY the JSON blob"* to the prompt reduced this.
  - **Wrong argument name** (e.g. `{"temp": 25}` instead of `{"celsius": 25}`), which makes the tool raise a validation error.
  - **Wrong text passed to `count_words` / `reverse_string`**: the model may include the quotes or the whole question instead of only the target text, so the count or reversal is slightly off.
  - **Numbers as strings** (`"25"` instead of `25`). LangChain usually converts these automatically because the argument is typed `float`.
- **Why it happens:** phi3 is a small model that was **not fine-tuned for tool calling**. It only follows our prompt instructions, so it is less reliable than models with native tool calling. Clear docstrings, exact argument names, and strict output instructions make a big difference.